In [6]:
import pandas as pd
import numpy as np
import re

# ── 0. 데이터 로드 ─────────────────────────────────────────────
df = pd.read_csv("PSID_with_TARGET.csv", low_memory=False)

# ── 1. 건강 · 금융 지표 컬럼 목록 정의 ────────────────────────────

# (1) 건강 수치형
health_num = [
    "BMI",       # 체질량지수
    "HOSP_NUM"   # 입·내원 횟수
]

# (2) 건강 이진형 (0/1 진단 여부)
health_bin = [
    c for c in df.columns
    if c.startswith(("CCON_","COVID_","DMNT_","DEPR_","DISA_"))
       and set(df[c].dropna().unique()) <= {0,1}
]

# (3) 건강 서열형
health_ord = [c for c in ["GHLTH_STAT","GHLTH_CHNG"] if c in df.columns]

# (4) 금융 수치형
fin_num = [
    "FINC_TOT_RD",    # 가구 총소득(실질)
    "EARN_TOT_ND",    # 소득 합계
    "FAM_SIZE_CHI",   # 자녀 수
    "FAM_SIZE"        # 가족 규모
]

# (5) 금융 이진형
fin_bin = [
    c for c in ["EMP_STAT_1M","FLAG_OWN_REALTY","FLAG_OWN_CAR"]
    if c in df.columns
]

# (6) 금융 범주형
fin_cat = [
    c for c in ["EDU_LEVEL","OCC_2010C_1M","GEO_REGION","NAME_INCOME_TYPE"]
    if c in df.columns
]

# 타깃
target = "TARGET"

# ── 2. 수치형 변수 상관계수 (Spearman) ───────────────────────────
num_vars = health_num + fin_num + [target]
corr = df[num_vars].corr(method="spearman")[target].sort_values(ascending=False).round(3)

print("=== 수치형 변수 × TARGET 상관계수 (Spearman) ===")
print(corr.to_frame(name="spearman_corr"))

# ── 3. 범주형 변수별 TARGET 평균 비교 ────────────────────────────
cat_vars = health_bin + health_ord + fin_bin + fin_cat

print("\n=== 범주형 변수별 TARGET(부실) 평균 ===")
for col in cat_vars:
    grp = df.groupby(col)[target]
    summary = pd.DataFrame({
        "count": grp.size(),
        "target_rate": grp.mean().round(3)
    }).sort_index()
    print(f"\n-- {col} --")
    print(summary)

# ── 4. 걷기·외출 관련 변수 자동 탐색 ────────────────────────────
# 정확히 ADL_Q5*, ADL_Q6* 항목만 잡아내도록
pattern = re.compile(r"^ADL_Q[56]_(ANY|HLP)$")
walk_cols = [c for c in df.columns if pattern.match(c)]

print("\n=== 걷기·외출 관련 변수 ===")
print(walk_cols if walk_cols else "없음")

# ── 5. 상관행렬 CSV로 저장 (선택) ───────────────────────────────
# df[num_vars].corr(method="spearman").to_csv("corr_matrix_health_financial.csv")


=== 수치형 변수 × TARGET 상관계수 (Spearman) ===
              spearman_corr
TARGET                1.000
BMI                  -0.011
HOSP_NUM             -0.011
EARN_TOT_ND          -0.021
FINC_TOT_RD          -0.048
FAM_SIZE_CHI         -0.058
FAM_SIZE             -0.070

=== 범주형 변수별 TARGET(부실) 평균 ===

-- CCON_ARTH_DIAG_ANY --
                    count  target_rate
CCON_ARTH_DIAG_ANY                    
0                   13397        0.018
1                    3058        0.021

-- CCON_ARTH_CARE --
                count  target_rate
CCON_ARTH_CARE                    
0               15050        0.018
1                1405        0.028

-- CCON_ASTH_DIAG_ANY --
                    count  target_rate
CCON_ASTH_DIAG_ANY                    
0                   14100        0.017
1                    2355        0.026

-- CCON_ASTH_CARE --
                count  target_rate
CCON_ASTH_CARE                    
0               14983        0.018
1                1472        0.024

-- CCON_CANC_DIA

In [7]:
data = pd.read_csv("./PSID_with_TARGET.csv")

In [ ]:
data['']

In [2]:
import tensorflow as tf 
from tensorflow.python.client import device_lib

print(device_lib.list_local_devices() )

2025-07-24 01:52:58.492602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 01:52:58.913742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753321979.064259      30 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753321979.110904      30 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753321979.438594      30 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 845455464186387235
xla_global_id: -1
]


W0000 00:00:1753321984.677047      30 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [15]:
# health_pd_demo.py  (v2: PCA 방향 보정 포함)  ───────────────────────────────────
"""
▷ improved_health_model.py(학습 로직) + Gradio UI 통합
▷ 처음 실행 시 학습·저장, 이후엔 바로 로드
▷ ADL·IADL 등 ‘값이 커질수록 위험’ 변수와 HEALTH_INDEX 방향을 자동 정렬
"""

import os, warnings, joblib, json, numpy as np, pandas as pd
import gradio as gr
from pathlib import Path

warnings.filterwarnings("ignore")

# ── 경로 설정 ─────────────────────────────────────────────────────────
DATA_CSV  = Path("./psid_with_target_prob.csv")   # 학습 데이터
MODEL_PKL = Path("./pd_models.pkl")               # 모델·전처리 번들

# ── 1. 학습 & 번들 저장 (처음 한 번) ─────────────────────────────────
if not MODEL_PKL.exists():
    print("🛠️  모델 없음 → 학습 시작 …")
    from sklearn.impute        import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model  import ElasticNetCV
    from sklearn.model_selection import KFold
    from catboost              import CatBoostRegressor, Pool

    # 1-1 데이터 로드
    df = pd.read_csv(DATA_CSV, low_memory=True)
    assert "TARGET" in df.columns, "TARGET 열이 필요합니다."
    y = df["TARGET"].astype(float).values

    AGE_COL = next((c for c in ["DEMO_AGE_GEN","DEMO_AGE_REP"] if c in df.columns), None)
    age = df[AGE_COL].astype(float).values if AGE_COL else np.zeros(len(df))

    fin_cols = [c for c in ["FINC_TOT_RDF","WLTH_TOT_NET_RDF","WLTH_TOT_DEB_RDF",
                            "EXPN_TOT_RDF","EXPN_HLTH_TOT_RDF"] if c in df.columns]
    health_base = [c for c in ["GHLTH_STAT","HOSP_ANY","HOSP_NUM",
                               "DEP_SCORE_TOT","ADL_SUM_TOT","IADL_SUM_TOT","BMI"] if c in df.columns]

    # 1-2 파생 변수 ────────────────────────────────────────────────
    def sigmoid(z): z=np.clip(z, -40, 40); return 1/(1+np.exp(-z))

    # (1) 의료비 부담률
    if {"EXPN_HLTH_TOT_RDF","FINC_TOT_RDF"} <= set(df.columns):
        df["HLTH_COST_RATIO"] = df["EXPN_HLTH_TOT_RDF"] / (df["FINC_TOT_RDF"].abs()+1)
    else:
        df["HLTH_COST_RATIO"] = np.nan

    # (2) ADL/IADL 플래그
    if "ADL_SUM_TOT"  in df.columns: df["ADL_FLAG"]  = (df["ADL_SUM_TOT"].fillna(0) > 0).astype(int)
    if "IADL_SUM_TOT" in df.columns: df["IADL_FLAG"] = (df["IADL_SUM_TOT"].fillna(0) > 0).astype(int)

    # (3) HEALTH_INDEX (PCA + 방향 보정)
    health_cols = health_base + ["HLTH_COST_RATIO"]
    imp_h = SimpleImputer(strategy="median").fit(df[health_cols])
    H_std = StandardScaler().fit_transform(imp_h.transform(df[health_cols]))
    pca   = PCA(n_components=1, random_state=42).fit(H_std)
    h_score = pca.transform(H_std).ravel()          # PC1

    # ── ★ 방향 보정: ‘값이 클수록 위험’ 참조 변수와 부호 일치 ★ ──
    ref_vars = ["ADL_SUM_TOT","IADL_SUM_TOT","HOSP_NUM","DEP_SCORE_TOT"]
    sign = 1                                          # 기본값
    for v in ref_vars:
        if v in df.columns:
            corr = np.corrcoef(h_score, df[v].fillna(0))[0,1]
            sign = 1 if corr < 0 else -1              # 위험↑ 변수와 양(+)의 상관값이 되도록
            break
    h_score *= sign

    healthy = sigmoid((h_score - h_score.mean())/(h_score.std()+1e-9))
    df["HEALTH_INDEX"] = 1 - healthy                 # 위험값 (↑=나쁨)

    # (4) 상호작용
    df["HX_AGE"] = df["HEALTH_INDEX"] * age
    if "HOSP_ANY" in df.columns:
        df["HOSP_AGE"] = df["HOSP_ANY"].fillna(0) * age
    if "HLTH_COST_RATIO" in df.columns:
        df["HCR_AGE"] = df["HLTH_COST_RATIO"].fillna(0) * age

    # 1-3 최종 피처 세트
    health_extra = ["HLTH_COST_RATIO","HEALTH_INDEX",
                    "ADL_FLAG","IADL_FLAG","HX_AGE","HOSP_AGE","HCR_AGE"]
    health_all = [c for c in health_base + health_extra if c in df.columns]
    fin_feats  = fin_cols
    all_feats  = fin_cols + health_all

    # 1-4 모델 학습
    def logit(p): p=np.clip(p,1e-6,1-1e-6); return np.log(p/(1-p))
    y_log = logit(y)

    # (A) 재무 전용 ElasticNet
    imp_fin = SimpleImputer(strategy="median").fit(df[fin_feats])
    sc_fin  = StandardScaler().fit(imp_fin.transform(df[fin_feats]))
    X_fin   = sc_fin.transform(imp_fin.transform(df[fin_feats]))
    en      = ElasticNetCV(l1_ratio=[.1,.3,.5,.7,.9,1],
                           alphas=np.logspace(-4,0,40),
                           cv=5, max_iter=20000, random_state=42).fit(X_fin, y_log)

    # (B) 재무+건강 CatBoost
    imp_all = SimpleImputer(strategy="median").fit(df[all_feats])
    X_all   = imp_all.transform(df[all_feats])
    cb      = CatBoostRegressor(depth=6, learning_rate=0.06, iterations=2000,
                                l2_leaf_reg=6, loss_function="RMSE",
                                random_seed=42, verbose=False).fit(
                                    Pool(X_all, y_log))

    # 1-5 번들 저장
    joblib.dump({
        "version": "1.2",
        "fin_model":   en,
        "aug_model":   cb,
        "imp_fin":     imp_fin,
        "sc_fin":      sc_fin,
        "imp_all":     imp_all,
        "imp_h":       imp_h,
        "sc_h":        StandardScaler().fit(H_std),   # 건강 스케일러
        "pca":         pca,
        "fin_feats":   fin_feats,
        "all_feats":   all_feats,
        "health_base": health_base
    }, MODEL_PKL)
    print(f"✅ 학습 완료 → {MODEL_PKL}")

# ── 2. 번들 로드 ────────────────────────────────────────────────
bundle   = joblib.load(MODEL_PKL)
finM     = bundle["fin_model"]
augM     = bundle["aug_model"]
imp_fin  = bundle["imp_fin"];   sc_fin = bundle["sc_fin"]
imp_all  = bundle["imp_all"]
imp_h    = bundle["imp_h"];     sc_h   = bundle["sc_h"]; pca=bundle["pca"]
fin_feats= bundle["fin_feats"]; all_feats=bundle["all_feats"]
health_base = bundle["health_base"]

# ── 3. 예측 유틸 ────────────────────────────────────────────────
def make_features(user: dict) -> pd.DataFrame:
    """사용자 입력 dict → 1-row DataFrame(파생 포함)"""
    r = user.copy()

    # (1) HLTH_COST_RATIO
    r["HLTH_COST_RATIO"] = (r["EXPN_HLTH_TOT_RDF"] /
                            (abs(r["FINC_TOT_RDF"])+1)) if \
                            r.get("EXPN_HLTH_TOT_RDF") is not None else np.nan
    # (2) 플래그
    r["ADL_FLAG"]  = int(r.get("ADL_SUM_TOT", 0)  > 0)
    r["IADL_FLAG"] = int(r.get("IADL_SUM_TOT", 0) > 0)

    # (3) HEALTH_INDEX
    H_raw = np.array([[r.get(c, np.nan) for c in health_base + ["HLTH_COST_RATIO"]]])
    H_std_u = sc_h.transform(imp_h.transform(H_raw))
    # PCA 변환 후 sign 방향은 학습 때 이미 고정됨
    h_score_u = pca.transform(H_std_u).ravel()
    healthy_u = 1/(1+np.exp(-(h_score_u)))         # sigmoid
    r["HEALTH_INDEX"] = 1 - healthy_u
    # (4) 상호작용
    age = r.get("AGE", 0)
    r["HX_AGE"]   = r["HEALTH_INDEX"] * age
    r["HOSP_AGE"] = (r.get("HOSP_ANY", 0) or 0) * age
    r["HCR_AGE"]  = (r.get("HLTH_COST_RATIO", 0) or 0) * age

    return pd.DataFrame([r], columns=all_feats)

def predict_pd(user: dict):
    """dict 입력 → (PD(fin), PD(fin+health)) [%]"""
    row = make_features(user)

    # 재무 전용
    X_f = sc_fin.transform(imp_fin.transform(row[fin_feats]))
    pd_fin = 1/(1+np.exp(-finM.predict(X_f))) * 100

    # 재무+건강
    X_a = imp_all.transform(row[all_feats])
    pd_aug = 1/(1+np.exp(-augM.predict(X_a))) * 100
    return round(pd_fin[0], 4), round(pd_aug[0], 4)

# ── 4. Gradio UI ───────────────────────────────────────────────
with gr.Blocks(title="PD Demo") as demo:
    gr.Markdown("## 📊 PD 예측 데모 (단위 %)")

    with gr.Accordion("재무 정보", open=True):
        FINC_TOT_RDF      = gr.Number(label="연소득",      value=60000)
        WLTH_TOT_NET_RDF  = gr.Number(label="총자산",      value=200000)
        WLTH_TOT_DEB_RDF  = gr.Number(label="총부채",      value=30000)
        EXPN_TOT_RDF      = gr.Number(label="연 총지출",    value=32000)
        EXPN_HLTH_TOT_RDF = gr.Number(label="연 의료비",    value=1500)

    with gr.Accordion("건강 정보 (선택)", open=False):
        AGE            = gr.Number(label="나이(AGE)",      value=60)
        GHLTH_STAT     = gr.Number(label="자가건강(1~5)",   value=4)
        HOSP_ANY       = gr.Number(label="입원 여부(0/1)",  value=0)
        HOSP_NUM       = gr.Number(label="입원 횟수",       value=0)
        DEP_SCORE_TOT  = gr.Number(label="우울 점수(0~24)", value=5)
        ADL_SUM_TOT    = gr.Number(label="ADL 합계",        value=0)
        IADL_SUM_TOT   = gr.Number(label="IADL 합계",       value=0)
        BMI            = gr.Number(label="BMI",            value=22)

    btn = gr.Button("🚀 예측하기")
    out1 = gr.Number(label="PD – 재무 전용(%)")
    out2 = gr.Number(label="PD – 재무+건강(%)")

    inputs = [FINC_TOT_RDF, WLTH_TOT_NET_RDF, WLTH_TOT_DEB_RDF,
              EXPN_TOT_RDF, EXPN_HLTH_TOT_RDF,
              AGE, GHLTH_STAT, HOSP_ANY, HOSP_NUM, DEP_SCORE_TOT,
              ADL_SUM_TOT, IADL_SUM_TOT, BMI]

    def _wrap(*vals):
        keys = ["FINC_TOT_RDF","WLTH_TOT_NET_RDF","WLTH_TOT_DEB_RDF",
                "EXPN_TOT_RDF","EXPN_HLTH_TOT_RDF",
                "AGE","GHLTH_STAT","HOSP_ANY","HOSP_NUM","DEP_SCORE_TOT",
                "ADL_SUM_TOT","IADL_SUM_TOT","BMI"]
        user = {k: (v if v not in (None, "") else np.nan) for k, v in zip(keys, vals)}
        return predict_pd(user)

    btn.click(fn=_wrap, inputs=inputs, outputs=[out1, out2])

if __name__ == "__main__":
    demo.launch()


Running on local URL:  http://127.0.0.1:7867

To create a public link, set `share=True` in `launch()`.


In [2]:
# health_pd_demo.py  (v5: HEALTH_SCORE↑ = 양호 · HEALTH_RISK↑ = 취약) ──────────────
"""
▷ 한 번만 학습해 pd_models.pkl 저장, 이후 로드
▷ 모든 금액 입력은 USD 기준
▷ HEALTH_SCORE(0~1, ↑=건강 양호)만 UI에 표시
"""

import os, warnings, joblib, numpy as np, pandas as pd, gradio as gr
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 파일 경로 ─────────────────────────────────────────────────────────
DATA_CSV  = Path("./psid_with_target_prob.csv")
MODEL_PKL = Path("./pd_models_4.pkl")

# ── 보조 함수 ─────────────────────────────────────────────────────────
def sigmoid(z): z = np.clip(z, -40, 40); return 1 / (1 + np.exp(-z))
def logit(p):  p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

# ── 1. 학습(최초 실행 시) ─────────────────────────────────────────────
if not MODEL_PKL.exists():
    print("🛠️  모델 없음 → 학습 시작 …")
    from sklearn.impute        import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    from sklearn.linear_model  import ElasticNetCV
    from catboost              import CatBoostRegressor, Pool

    # 1-1 데이터
    df = pd.read_csv(DATA_CSV, low_memory=True)
    y  = df["TARGET"].astype(float).values

    AGE_COL = next((c for c in ["DEMO_AGE_GEN", "DEMO_AGE_REP"]
                    if c in df.columns), None)
    age = df[AGE_COL].astype(float).values if AGE_COL else np.zeros(len(df))

    fin_cols = [c for c in [
        "FINC_TOT_RDF", "WLTH_TOT_NET_RDF", "WLTH_TOT_DEB_RDF",
        "EXPN_TOT_RDF", "EXPN_HLTH_TOT_RDF"] if c in df.columns]

    health_base = [c for c in [
        "GHLTH_STAT", "HOSP_ANY", "HOSP_NUM", "DEP_SCORE_TOT",
        "ADL_SUM_TOT", "IADL_SUM_TOT", "BMI"] if c in df.columns]

    # 1-2 파생
    df["HLTH_COST_RATIO"] = df["EXPN_HLTH_TOT_RDF"] / (
        df["FINC_TOT_RDF"].abs() + 1)

    df["ADL_FLAG"]  = (df["ADL_SUM_TOT"].fillna(0)  > 0).astype(int)
    df["IADL_FLAG"] = (df["IADL_SUM_TOT"].fillna(0) > 0).astype(int)

    # HEALTH_SCORE (↑=양호) -- PCA + 시그모이드
    health_cols = health_base + ["HLTH_COST_RATIO"]
    imp_h = SimpleImputer(strategy="median").fit(df[health_cols])
    H_std = StandardScaler().fit_transform(imp_h.transform(df[health_cols]))
    pca   = PCA(n_components=1, random_state=42).fit(H_std)
    h_raw = pca.transform(H_std).ravel()

    # 방향: ADL_SUM_TOT와 음(–) 상관이 되도록
    sign = 1 if np.corrcoef(h_raw, df["ADL_SUM_TOT"].fillna(0))[0, 1] < 0 else -1
    HEALTH_SCORE = sigmoid(sign * h_raw)           # ↑ = 양호
    df["HEALTH_SCORE"] = HEALTH_SCORE
    df["HEALTH_RISK"]  = 1 - HEALTH_SCORE          # ↑ = 취약 (모델 피처용)

    # 상호작용
    df["HX_AGE"]   = df["HEALTH_RISK"] * age
    df["HOSP_AGE"] = df["HOSP_ANY"].fillna(0) * age
    df["HCR_AGE"]  = df["HLTH_COST_RATIO"].fillna(0) * age

    # 1-3 피처 세트
    health_extra = [
        "HLTH_COST_RATIO", "HEALTH_RISK",
        "ADL_FLAG", "IADL_FLAG", "HX_AGE", "HOSP_AGE", "HCR_AGE"]
    health_all   = [c for c in health_base + health_extra if c in df.columns]
    fin_feats    = fin_cols
    all_feats    = fin_cols + health_all

    # 1-4 모델
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    imp_fin = SimpleImputer(strategy="median").fit(df[fin_feats])
    sc_fin  = StandardScaler().fit(imp_fin.transform(df[fin_feats]))
    X_fin   = sc_fin.transform(imp_fin.transform(df[fin_feats]))

    en = ElasticNetCV(
        l1_ratio=[.1, .3, .5, .7, .9, 1],
        alphas=np.logspace(-4, 0, 40),
        cv=5, max_iter=20000, random_state=42
    ).fit(X_fin, logit(y))

    imp_all = SimpleImputer(strategy="median").fit(df[all_feats])
    X_all   = imp_all.transform(df[all_feats])

    cb = CatBoostRegressor(
        depth=6, learning_rate=0.06, iterations=2000,
        l2_leaf_reg=6, loss_function="RMSE",
        random_seed=42, verbose=False
    ).fit(Pool(X_all, logit(y)))

    # 1-5 번들 저장
    joblib.dump({
        "fin_model": en, "aug_model": cb,
        "imp_fin": imp_fin, "sc_fin": sc_fin,
        "imp_all": imp_all,
        "imp_h": imp_h, "sc_h": StandardScaler().fit(H_std), "pca": pca,
        "fin_feats": fin_feats, "all_feats": all_feats,
        "health_base": health_base
    }, MODEL_PKL)
    print("✅ 학습 완료 →", MODEL_PKL)

# ── 2. 번들 로드 ───────────────────────────────────────────────
b = joblib.load(MODEL_PKL)
finM, augM           = b["fin_model"], b["aug_model"]
imp_fin, sc_fin      = b["imp_fin"], b["sc_fin"]
imp_all, imp_h, sc_h = b["imp_all"], b["imp_h"], b["sc_h"]
pca                  = b["pca"]
fin_feats, all_feats, health_base = b["fin_feats"], b["all_feats"], b["health_base"]

# ── 3. 예측 유틸 ───────────────────────────────────────────────
def make_features(u: dict) -> pd.DataFrame:
    r = u.copy()
    r["HLTH_COST_RATIO"] = r["EXPN_HLTH_TOT_RDF"] / (abs(r["FINC_TOT_RDF"])+1)
    r["ADL_FLAG"]  = int(r.get("ADL_SUM_TOT",0)  > 0)
    r["IADL_FLAG"] = int(r.get("IADL_SUM_TOT",0) > 0)

    H = np.array([[r.get(c, np.nan) for c in health_base + ["HLTH_COST_RATIO"]]])
    H_std_u = sc_h.transform(imp_h.transform(H))
    h_raw_u = pca.transform(H_std_u).ravel()
    HEALTH_SCORE_u = sigmoid(h_raw_u)       # ↑ = 양호
    r["HEALTH_SCORE"] = HEALTH_SCORE_u
    r["HEALTH_RISK"]  = 1 - HEALTH_SCORE_u  # ↑ = 취약

    age = r.get("AGE",0)
    r["HX_AGE"]   = r["HEALTH_RISK"] * age
    r["HOSP_AGE"] = (r.get("HOSP_ANY",0) or 0) * age
    r["HCR_AGE"]  = (r.get("HLTH_COST_RATIO",0) or 0) * age

    return pd.DataFrame([r], columns=all_feats + ["HEALTH_SCORE"])

def predict_pd(u: dict):
    row = make_features(u)

    pd_fin = 1/(1+np.exp(-finM.predict(
        sc_fin.transform(imp_fin.transform(row[fin_feats])))))*100
    pd_aug = 1/(1+np.exp(-augM.predict(
        imp_all.transform(row[all_feats]))))*100
    score  = row["HEALTH_SCORE"].iloc[0]
    return round(pd_fin[0],4), round(pd_aug[0],4), round(float(score),4)

# ── 4. Gradio UI (USD) ─────────────────────────────────────────
with gr.Blocks(title="PD Demo (USD)") as demo:
    gr.Markdown("## 📊 PD 예측  —  모든 금액은 **USD** 기준")

    with gr.Accordion("재무 정보 (USD)", open=True):
        FINC_TOT_RDF      = gr.Number(label="Annual Income", value=50000)
        WLTH_TOT_NET_RDF  = gr.Number(label="Total Assets",  value=120000)
        WLTH_TOT_DEB_RDF  = gr.Number(label="Total Debt",    value=30000)
        EXPN_TOT_RDF      = gr.Number(label="Annual Spending", value=35000)
        EXPN_HLTH_TOT_RDF = gr.Number(label="Annual Medical Spending", value=2500)

    with gr.Accordion("건강 정보 (선택)", open=False):
        AGE            = gr.Number(label="Age", value=60)
        GHLTH_STAT     = gr.Number(label="Self-rated Health (1~5)", value=4)
        HOSP_ANY       = gr.Number(label="Hospitalized? (0/1)", value=0)
        HOSP_NUM       = gr.Number(label="Times Hospitalized",  value=0)
        DEP_SCORE_TOT  = gr.Number(label="Depression Score (0~24)", value=5)
        ADL_SUM_TOT    = gr.Number(label="ADL Sum",  value=0)
        IADL_SUM_TOT   = gr.Number(label="IADL Sum", value=0)
        BMI            = gr.Number(label="BMI",      value=22)

    btn = gr.Button("🚀 Predict")
    out_fin   = gr.Number(label="PD – Finance Only (%)")
    out_aug   = gr.Number(label="PD – Finance + Health (%)")
    out_score = gr.Number(label="HEALTH_SCORE (0~1, ↑ = 건강)")

    keys = ["FINC_TOT_RDF","WLTH_TOT_NET_RDF","WLTH_TOT_DEB_RDF",
            "EXPN_TOT_RDF","EXPN_HLTH_TOT_RDF",
            "AGE","GHLTH_STAT","HOSP_ANY","HOSP_NUM","DEP_SCORE_TOT",
            "ADL_SUM_TOT","IADL_SUM_TOT","BMI"]

    def _wrap(*vals):
        user = {k:(v if v not in (None,"") else np.nan)
                for k, v in zip(keys, vals)}
        return predict_pd(user)

    btn.click(_wrap, inputs=[
        FINC_TOT_RDF, WLTH_TOT_NET_RDF, WLTH_TOT_DEB_RDF,
        EXPN_TOT_RDF, EXPN_HLTH_TOT_RDF,
        AGE, GHLTH_STAT, HOSP_ANY, HOSP_NUM, DEP_SCORE_TOT,
        ADL_SUM_TOT, IADL_SUM_TOT, BMI],
        outputs=[out_fin, out_aug, out_score])

if __name__ == "__main__":
    demo.launch()


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [26]:
# build_pd_models_legacy.py  ──────────────────────────────────────────────
"""
원본 improved_health_model.py 로직 *그대로* 재현하여
  1) 재무 전용  ElasticNetCV  (p_fin)
  2) 재무+건강   CatBoostRegressor (p_aug)
두 모델과 모든 전처리·피처 목록을 번들(pd_models_legacy.pkl)로 저장합니다.

실행:  python build_pd_models_legacy.py
"""

import os, joblib, json, warnings, numpy as np, pandas as pd
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNetCV
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")

# ── 경로 및 상수 ────────────────────────────────────────────
DATA_CSV   = Path("./psid_with_target_prob.csv")
MODEL_PKL  = Path("./pd_models_legacy.pkl")       # ← 새로 저장할 번들

# ── 보조 함수 ──────────────────────────────────────────────
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def sigmoid(z):
    z = np.clip(z, -40, 40)
    return 1 / (1 + np.exp(-z))

# ── 1. 데이터 로드 ─────────────────────────────────────────
df = pd.read_csv(DATA_CSV, low_memory=True)
assert "TARGET" in df.columns, "TARGET 열이 없습니다."
y = df["TARGET"].astype(float).values

AGE_COL = (
    "DEMO_AGE_GEN" if "DEMO_AGE_GEN" in df.columns else
    ("DEMO_AGE_REP" if "DEMO_AGE_REP" in df.columns else None)
)
age = df[AGE_COL].astype(float).values if AGE_COL else np.zeros(len(df))

# ── 2. 피처 기본 목록 ──────────────────────────────────────
fin_cols = [c for c in [
    "FINC_TOT_RDF", "WLTH_TOT_NET_RDF", "WLTH_TOT_DEB_RDF",
    "EXPN_TOT_RDF", "EXPN_HLTH_TOT_RDF"] if c in df.columns]

health_base = [c for c in [
    "GHLTH_STAT", "HOSP_ANY", "HOSP_NUM", "DEP_SCORE_TOT",
    "ADL_SUM_TOT", "IADL_SUM_TOT", "BMI"] if c in df.columns]

# ── 3. 파생 변수 (원본 공식 그대로) ────────────────────────
df["HLTH_COST_RATIO"] = (
    df["EXPN_HLTH_TOT_RDF"] /
    (df["FINC_TOT_RDF"].abs() + 1)
    if {"EXPN_HLTH_TOT_RDF", "FINC_TOT_RDF"} <= set(df.columns)
    else np.nan
)

df["ADL_FLAG"]  = (df["ADL_SUM_TOT"].fillna(0)  > 0).astype(int) if "ADL_SUM_TOT"  in df.columns else 0
df["IADL_FLAG"] = (df["IADL_SUM_TOT"].fillna(0) > 0).astype(int) if "IADL_SUM_TOT" in df.columns else 0

# HEALTH_INDEX 계산(부호 교정 없이 그대로)
health_cols = health_base + ["HLTH_COST_RATIO"]
imp_h = SimpleImputer(strategy="median").fit(df[health_cols])
H_std = StandardScaler().fit_transform(imp_h.transform(df[health_cols]))
pca   = PCA(n_components=1, random_state=42).fit(H_std)
h_score = pca.transform(H_std).ravel()
healthy      = sigmoid((h_score - H_std.mean()) / (H_std.std() + 1e-9))
df["HEALTH_INDEX"] = 1 - healthy           # ↑ = 위험  (원본과 동일)

# 상호작용
df["HX_AGE"]  = df["HEALTH_INDEX"] * age
df["HOSP_AGE"] = df["HOSP_ANY"].fillna(0) * age if "HOSP_ANY" in df.columns else 0
df["HCR_AGE"]  = df["HLTH_COST_RATIO"].fillna(0) * age

# ── 4. 최종 피처 세트 ─────────────────────────────────────
features_fin  = fin_cols
features_hlth = [
    *health_base,
    "HLTH_COST_RATIO", "HEALTH_INDEX",
    "ADL_FLAG", "IADL_FLAG",
    "HX_AGE", "HOSP_AGE", "HCR_AGE"
]
features_hlth  = [c for c in features_hlth if c in df.columns]
features_all   = features_fin + features_hlth

# ── 5. 재무 전용 ElasticNetCV ─────────────────────────────
imp_fin = SimpleImputer(strategy="median").fit(df[features_fin])
sc_fin  = StandardScaler().fit(imp_fin.transform(df[features_fin]))
X_fin   = sc_fin.transform(imp_fin.transform(df[features_fin]))

en = ElasticNetCV(
    l1_ratio=[.1, .3, .5, .7, .9, 1],
    alphas=np.logspace(-4, 0, 40),
    cv=5, max_iter=20000,
    random_state=42
).fit(X_fin, logit(y))

# ── 6. 재무+건강 CatBoostRegressor ───────────────────────
imp_all = SimpleImputer(strategy="median").fit(df[features_all])
X_all   = imp_all.transform(df[features_all])

cb = CatBoostRegressor(
    depth=6, learning_rate=0.06, iterations=2000,
    l2_leaf_reg=6, loss_function="RMSE",
    random_seed=42, verbose=False
).fit(Pool(X_all, logit(y)))

# ── 7. 번들 저장 ─────────────────────────────────────────
bundle = {
    "fin_model":   en,
    "aug_model":   cb,
    "imp_fin":     imp_fin,
    "sc_fin":      sc_fin,
    "imp_all":     imp_all,
    "imp_h":       imp_h,
    "sc_h":        StandardScaler().fit(H_std),  # 표준화(μ,σ) 보존
    "pca":         pca,
    "fin_feats":   features_fin,
    "all_feats":   features_all,
    "health_base": health_base
}
joblib.dump(bundle, MODEL_PKL)
print(f"✅ 번들 저장 완료 → {MODEL_PKL}")

# (선택) 모델 예측값 npz 파일로 저장해 두기
p_fin_full = 1 / (1 + np.exp(-en.predict(X_fin)))
p_aug_full = 1 / (1 + np.exp(-cb.predict(X_all)))
np.savez("model_preds_legacy.npz",
         p_fin=p_fin_full,
         p_aug=p_aug_full)
print("예측값 'model_preds_legacy.npz' 저장 ✅")


✅ 번들 저장 완료 → pd_models_legacy.pkl
예측값 'model_preds_legacy.npz' 저장 ✅


In [4]:
# pd_gradio_app.py  ────────────────────────────────────────────────────────
"""
📊 PD 예측 데모  (USD 단위)

좌측  ▶ 재무 정보  
우측  ▶ 건강 정보  
하단  ▶ 예측 결과 (재무-PD, 재무+건강-PD)
"""

import numpy as np, pandas as pd, joblib, gradio as gr

# ── 1. 모델·전처리 로드 ──────────────────────────────────────────
b_fin = joblib.load("./fin_model.pkl")
b_aug = joblib.load("./aug_model.pkl")

fin_model, imp_fin, sc_fin, fin_feats = (
    b_fin["model"], b_fin["imp"], b_fin["sc"], b_fin["features"])

aug_model   = b_aug["model"]
imp_all     = b_aug["imp_all"]
imp_h, sc_h, pca = b_aug["imp_h"], b_aug["sc_h"], b_aug["pca"]
all_feats   = b_aug["features"]
health_base = b_aug["health_base"]

# ── 2. 파생·예측 함수 ──────────────────────────────────────────
def sigmoid(z): z=np.clip(z,-40,40); return 1/(1+np.exp(-z))

def make_features(u: dict) -> pd.DataFrame:
    r = u.copy()
    r["HLTH_COST_RATIO"] = r["EXPN_HLTH_TOT_RDF"] / (abs(r["FINC_TOT_RDF"])+1)
    r["ADL_FLAG"]  = int(r.get("ADL_SUM_TOT",0)  > 0)
    r["IADL_FLAG"] = int(r.get("IADL_SUM_TOT",0) > 0)

    H = np.array([[r.get(c,np.nan) for c in health_base + ["HLTH_COST_RATIO"]]])
    H_std = sc_h.transform(imp_h.transform(H))
    r["HEALTH_INDEX"] = 1 - sigmoid(pca.transform(H_std).ravel())

    age = r.get("AGE",0)
    r["HX_AGE"]   = r["HEALTH_INDEX"] * age
    r["HOSP_AGE"] = (r.get("HOSP_ANY",0) or 0) * age
    r["HCR_AGE"]  = (r["HLTH_COST_RATIO"] or 0) * age
    return pd.DataFrame([r], columns=all_feats)

def predict_pd(user):
    row = make_features(user)
    Xf = sc_fin.transform(imp_fin.transform(row[fin_feats]))
    pd_fin = 1/(1+np.exp(-fin_model.predict(Xf))) * 100
    Xa = imp_all.transform(row[all_feats])
    pd_aug = 1/(1+np.exp(-aug_model.predict(Xa))) * 100
    return round(pd_fin[0],4), round(pd_aug[0],4)

# ── 3. 기본값 (실제 사례) ─────────────────────────────────────
defaults = dict(
    FINC_TOT_RDF      = 358_983.9051,
    WLTH_TOT_NET_RDF  = 4_596_312.164,
    WLTH_TOT_DEB_RDF  = 107_906.5476,
    EXPN_TOT_RDF      = 54_122.96976,
    EXPN_HLTH_TOT_RDF = 16_022.34176,
    AGE            = 62,
    GHLTH_STAT     = 4,
    HOSP_ANY       = 0,
    HOSP_NUM       = 0,
    DEP_SCORE_TOT  = 12,
    ADL_SUM_TOT    = 0,
    IADL_SUM_TOT   = 0,
    BMI            = 18.55,
)

# ── 4. Gradio UI  (재무 왼쪽 · 건강 오른쪽 · 결과 아래) ─────────────
with gr.Blocks(title="PD Predictor (USD)") as demo:
    gr.Markdown("### 🏦→🩺  PD 예측\n\n"
                "실제 고객 데이터를 그대로 사용했습니다. 원하는 값을 바꿔보세요!")

    with gr.Row():
        # ── 좌측: 재무 ───────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("#### 💰 재무 정보 (USD)")
            FINC_TOT_RDF      = gr.Number(label="Annual Income",               value=defaults["FINC_TOT_RDF"])
            WLTH_TOT_NET_RDF  = gr.Number(label="Total Assets",                value=defaults["WLTH_TOT_NET_RDF"])
            WLTH_TOT_DEB_RDF  = gr.Number(label="Total Debt",                  value=defaults["WLTH_TOT_DEB_RDF"])
            EXPN_TOT_RDF      = gr.Number(label="Annual Spending",             value=defaults["EXPN_TOT_RDF"])
            EXPN_HLTH_TOT_RDF = gr.Number(label="Annual Medical Spending",     value=defaults["EXPN_HLTH_TOT_RDF"])

        # ── 우측: 건강 ───────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("#### 🩺 건강 정보")
            AGE            = gr.Number(label="Age",                       value=defaults["AGE"])
            GHLTH_STAT     = gr.Number(label="Self-rated Health (1~5)",   value=defaults["GHLTH_STAT"])
            HOSP_ANY       = gr.Number(label="Hospitalized? (0/1)",       value=defaults["HOSP_ANY"])
            HOSP_NUM       = gr.Number(label="Times Hospitalized",        value=defaults["HOSP_NUM"])
            DEP_SCORE_TOT  = gr.Number(label="Depression Score (0~24)",   value=defaults["DEP_SCORE_TOT"])
            ADL_SUM_TOT    = gr.Number(label="ADL Sum",                   value=defaults["ADL_SUM_TOT"])
            IADL_SUM_TOT   = gr.Number(label="IADL Sum",                  value=defaults["IADL_SUM_TOT"])
            BMI            = gr.Number(label="BMI",                       value=defaults["BMI"])

    # ── 결과 + 버튼 아래 배치 ─────────────────────────────────
    btn = gr.Button("🚀  Predict", variant="primary")
    with gr.Row():
        out_fin = gr.Number(label="체납 확률 – Finance Only (%)")
        out_aug = gr.Number(label="체납 확률 – Finance + Health (%)")

    # ── 연결 ────────────────────────────────────────────────
    inputs = [FINC_TOT_RDF, WLTH_TOT_NET_RDF, WLTH_TOT_DEB_RDF,
              EXPN_TOT_RDF, EXPN_HLTH_TOT_RDF,
              AGE, GHLTH_STAT, HOSP_ANY, HOSP_NUM, DEP_SCORE_TOT,
              ADL_SUM_TOT, IADL_SUM_TOT, BMI]

    keys = ["FINC_TOT_RDF","WLTH_TOT_NET_RDF","WLTH_TOT_DEB_RDF",
            "EXPN_TOT_RDF","EXPN_HLTH_TOT_RDF",
            "AGE","GHLTH_STAT","HOSP_ANY","HOSP_NUM","DEP_SCORE_TOT",
            "ADL_SUM_TOT","IADL_SUM_TOT","BMI"]

    btn.click(lambda *v: predict_pd({k:v for k,v in zip(keys,v)}),
              inputs=inputs, outputs=[out_fin, out_aug])

if __name__ == "__main__":
    demo.launch()


Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
